In [8]:
# ============================================================
# PREPROCESSING DATA FATSECRET UNTUK SISTEM DASH
# ============================================================

import pandas as pd
import re

INPUT_FILE = "fatsecret_mapped_8sheets.xlsx"
OUTPUT_FILE = "Data_Nutrisi_Selected_Clean.xlsx"


# ============================================================
# FUNGSI PEMBANTU UNTUK FORMAT TEKS
# ============================================================

def format_ingredients(text):
    if pd.isna(text):
        return ""
    items = [x.strip() for x in str(text).split("|") if x.strip()]
    return "\n".join(items)

def format_instructions(text):
    if pd.isna(text):
        return ""
    steps = [x.strip() for x in str(text).split("|") if x.strip()]
    return "\n".join([f"{i+1}. {step}" for i, step in enumerate(steps)])

# ============================================================
# FEATURE SELECTION (AMBIL KOLOM-KOLOM YANG DIPERLUKAN)
# ============================================================

def preprocess_selected_columns():
    # === Baca semua sheet ===
    df_all = pd.read_excel(INPUT_FILE, sheet_name=None)

    # Definisi kolom resep
    recipe_cols = {
        "title": "nama_menu",
        "recipeYield": "porsi",
        "ingredients": "bahan-bahan",
        "instructions": "cara_pembuatan",
        "cookTime": "waktu_memasak",
        "kalori": "kalori",
        "protein": "protein",
        "lemak": "lemak",
        "lemak_jenuh": "lemak_jenuh",
        "kolesterol": "kolesterol",
        "karbohidrat": "karbohidrat",
        "sodium": "natrium",
        "kalium": "kalium",
        "serat": "serat",
    }

    # Definisi kolom buah
    buah_cols = {
        "nama_buah": "nama_buah",
        "ukuran_porsi": "porsi",
        "kalori": "kalori",
        "protein": "protein",
        "lemak": "lemak",
        "lemak_jenuh": "lemak_jenuh",
        "kolesterol": "kolesterol",
        "karbohidrat": "karbohidrat",
        "sodium": "natrium",
        "kalium": "kalium",
        "serat": "serat",     
    }

    selected_sheets = {}

    # Proses menu yang memiliki resep  
    for sheet in ["sarapan", "makan_siang", "makan_malam", "sayur", "jus_smoothie"]:
        if sheet in df_all:
            df = df_all[sheet].copy()
            available = [c for c in recipe_cols.keys() if c in df.columns]
            df_selected = df[available].rename(columns=recipe_cols)

            # Format bahan & cara pembuatan
            if "bahan-bahan" in df_selected.columns:
                df_selected["bahan-bahan"] = df_selected["bahan-bahan"].apply(format_ingredients)
            if "cara_pembuatan" in df_selected.columns:
                df_selected["cara_pembuatan"] = df_selected["cara_pembuatan"].apply(format_instructions)

            selected_sheets[sheet] = df_selected

    # Proses buah
    if "buah" in df_all:
        df_buah = df_all["buah"].copy()
        available = [c for c in buah_cols.keys() if c in df_buah.columns]
        df_buah_selected = df_buah[available].rename(columns=buah_cols)
        selected_sheets["buah"] = df_buah_selected

    # Tambahkan susu dan nasi
    for sheet in ["susu", "nasi"]:
        if sheet in df_all:
            df_simple = df_all[sheet].copy()

            # pastikan kolom numerik bersih
            for col in [
                "kalori", "protein", "lemak", "lemak_jenuh",
                "kolesterol", "karbohidrat", "natrium",
                "kalium", "serat", "kalsium"
            ]:
                if col in df_simple.columns:
                    df_simple[col] = df_simple[col].apply(clean_numeric).fillna(0)

            selected_sheets[sheet] = df_simple

    return selected_sheets

# ============================================================
# KONVERSI KE NUMERIC
# ============================================================

def clean_numeric(value):
    """Bersihkan angka dari string (misal '136,0 kkal' → 136.0)."""
    if pd.isna(value):
        return None
    value = str(value).lower()
    value = re.sub(r"[^\d,.-]", "", value)  # hapus huruf
    value = value.replace(",", ".")         # koma → titik
    try:
        return float(value)
    except:
        return None
    
# ============================================================
# DEDUPLIKASI BUAH (AMBIL PORSI TERBESAR)
# ============================================================

def deduplikasi_buah(df_buah):
    print("\n" + "="*60)
    print("DEDUPLIKASI BUAH (AMBIL PORSI TERBESAR)")
    print("="*60)
    
    sebelum = len(df_buah)
    
    # Sort by kalori (descending) lalu drop duplicates berdasarkan nama_buah
    # keep='first' akan otomatis ambil yang kalori tertinggi
    df_deduplicated = (df_buah
                       .sort_values('kalori', ascending=False)
                       .drop_duplicates(subset=['nama_buah'], keep='first')
                       .reset_index(drop=True))
    
    sesudah = len(df_deduplicated)
    dihapus = sebelum - sesudah
    
    print(f"\nHasil Deduplikasi:")
    print(f"   Sebelum: {sebelum} items")
    print(f"   Sesudah: {sesudah} items")
    print(f"   Dihapus: {dihapus} variasi porsi")
    
    print("\n" + "="*60)
    
    return df_deduplicated

def handle_and_save(selected_sheets):
    print("CEK & HANDLE MISSING VALUES")
    print("=" * 60)

    cleaned_sheets = {}

    nutrisi_cols = [
        "kalori", "natrium", "kalium", "serat", 
        "lemak", "lemak_total", "lemak_jenuh", "kolesterol",
        "protein", "karbohidrat"
    ]

    for sheet, df in selected_sheets.items():
        print(f"\nSheet: {sheet}")
        df_clean = df.copy()

        # CLEANING NUMERIK
        for col in nutrisi_cols:
            if col in df_clean.columns:
                df_clean[col] = df_clean[col].apply(clean_numeric)
                missing_rows = df_clean[df_clean[col].isna()]
                
                if len(missing_rows) > 0:
                    print(f"   - Kolom '{col}': {len(missing_rows)} missing → diisi 0")
                    df_clean[col] = df_clean[col].fillna(0)
        
        # HAPUS DUPLIKAT MURNI (semua kolom sama)
        sebelum_duplikat = len(df_clean)
        df_clean = df_clean.drop_duplicates(keep='first').reset_index(drop=True)
        duplikat_dihapus = sebelum_duplikat - len(df_clean)
        
        if duplikat_dihapus > 0:
            print(f"Duplikat murni dihapus: {duplikat_dihapus} baris")
        
        # DEDUPLIKASI KHUSUS BUAH (ambil porsi terbesar)
        if sheet == 'buah' and 'nama_buah' in df_clean.columns:
            df_clean = deduplikasi_buah(df_clean)

        cleaned_sheets[sheet] = df_clean

    # Simpan hasil akhir
    with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
        for sheet, df in cleaned_sheets.items():
            df.to_excel(writer, sheet_name=sheet, index=False)

    print("\n" + "="*60)
    print("DATA SUDAH DIBERSIHKAN & DISIMPAN")
    print("="*60)
    print(f"File output: {OUTPUT_FILE}")
    
    print(f"\nRINGKASAN FINAL:")
    for sheet, df in cleaned_sheets.items():
        print(f"   {sheet:<15} {len(df):>4} items")
    
    print("\nData siap digunakan untuk pembentukan arm!")
    print("="*60)

# ============================================================
# EKSEKUSI
# ============================================================

if __name__ == "__main__":
    selected_sheets = preprocess_selected_columns()
    handle_and_save(selected_sheets)


CEK & HANDLE MISSING VALUES

Sheet: sarapan

Sheet: makan_siang

Sheet: makan_malam

Sheet: sayur

Sheet: jus_smoothie

Sheet: buah
   - Kolom 'lemak_jenuh': 1 missing → diisi 0
   - Kolom 'kolesterol': 1 missing → diisi 0
Duplikat murni dihapus: 18 baris

DEDUPLIKASI BUAH (AMBIL PORSI TERBESAR)

Hasil Deduplikasi:
   Sebelum: 128 items
   Sesudah: 76 items
   Dihapus: 52 variasi porsi


Sheet: susu

Sheet: nasi

DATA SUDAH DIBERSIHKAN & DISIMPAN
File output: Data_Nutrisi_Selected_Clean.xlsx

RINGKASAN FINAL:
   sarapan           54 items
   makan_siang       54 items
   makan_malam       56 items
   sayur             38 items
   jus_smoothie      14 items
   buah              76 items
   susu               3 items
   nasi               2 items

Data siap digunakan untuk pembentukan arm!
